# ZTLF 03 — Baseline comparison

The question this notebook has to answer: how does my quality gate actually compare to the tools people already use?

Ground rules:
- Every tool gets the exact same logical rule set, written from documentation, never from the corruption plan.
- Every tool sees the same frozen, corrupted input.
- Every tool is scored the same way against the same ground truth, with naturally-defective cells excluded.
- If a tool can't express a constraint, that's a capability gap I record, not something to quietly work around.
- Runtime and rule-authoring effort get reported next to accuracy.

What I expect going in: given identical rules, my gate, Pandera, and Soda should land on identical detections. That's not a disappointing result — it's the correct one, and it should be reported plainly rather than dressed up. The contribution here isn't a smarter rule engine. What's actually interesting is where the tools differ in capability and cost.


In [ ]:
#@title Mount, imports, install baselines
import pathlib, sys, json, warnings, subprocess
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/Paper1")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
TABLES=PROJECT_ROOT/"outputs/tables"; FIGURES=PROJECT_ROOT/"outputs/figures"
METRICS=PROJECT_ROOT/"outputs/metrics"
for d in (TABLES,FIGURES,METRICS): d.mkdir(parents=True, exist_ok=True)

# Pinned baseline versions -- record these in the manuscript
BASELINE_PINS = {"pandera":"0.32.1","great_expectations":"1.19.1",
                 "soda-core":"4.19.0","duckdb":"latest"}
subprocess.run("pip -q install pandera==0.32.1 great_expectations==1.19.1 duckdb",
               shell=True)
print("installed:", BASELINE_PINS)

In [ ]:
#@title Load modules  (NO stub generation -- verifies the real module is present)
import sys, pathlib, importlib, pandas as pd, numpy as np, dataclasses, time

SRC = PROJECT_ROOT / "src"
required = ["ztlf_profiling.py", "ztlf_specs.py", "ztlf_corruption.py",
            "ztlf_plans.py", "ztlf_baselines.py"]
missing = [f for f in required if not (SRC / f).exists()]
if missing:
    raise SystemExit(
        f"Missing modules in {SRC}: {missing}\n"
        "Copy them from the Phase 1/2/3 packages into Drive. "
        "Do NOT generate replacements -- a stub silently drops most of the "
        "rule set and returns positional indices instead of row ids, which "
        "produces detections that match no ground truth (precision = recall = 0).")

import ztlf_profiling, ztlf_specs, ztlf_corruption, ztlf_plans, ztlf_baselines
for m in (ztlf_profiling, ztlf_specs, ztlf_corruption, ztlf_plans, ztlf_baselines):
    importlib.reload(m)

# --- integrity guard ---
assert isinstance(getattr(ztlf_baselines, "BASELINES", None), dict)
BASELINES = ztlf_baselines.BASELINES.copy()
if "ZTLF (ours)" in BASELINES:
    BASELINES["This work"] = BASELINES.pop("ZTLF (ours)")

_expected = {"This work", "Pandera", "GreatExpectations", "SodaCore/DuckDB"}
_have = set(BASELINES)
assert _expected <= _have, f"Missing tools: {_expected - _have}."
print("module integrity OK -- tools:", sorted(_have))

from ztlf_profiling import load_dataset
from ztlf_specs import *
from ztlf_corruption import (assign_row_ids, score_restricted,
                             natural_defect_keys_full, ROW_ID)
from ztlf_plans import PLANS, CONTAMINATION_RATES, seed_list
from ztlf_baselines import RuleSet, run_baselines

RAW = PROJECT_ROOT/"data/raw"
SPECS = {
  "bank_marketing_full": bank_spec("bank_marketing_full", str(RAW/"bank-full.csv")),
  "diabetes_130us": dataclasses.replace(DIABETES_SPEC, path=str(RAW/"diabetic_data.csv")),
  "online_retail_ii": online_retail_spec(str(RAW/"online_retail_II.csv")),
}
CLEAN={}
for n,s in SPECS.items():
    if pathlib.Path(s.path).exists():
        CLEAN[n]=assign_row_ids(load_dataset(s), n[:4])

In [ ]:
#@title Shared rule sets -- ONE source of truth for every tool
DIAB_DRUGS = {c: DIAB_DRUG_LEVELS for c in ("metformin","insulin","glipizide","glyburide")}

RULES = {
 "bank_marketing_full": RuleSet("bank_marketing_full",
   domains={"job":BANK_JOBS,"marital":BANK_MARITAL,"education":BANK_EDUCATION,
            "month":BANK_MONTH,"contact":BANK_CONTACT,"poutcome":BANK_POUTCOME,
            "default":BANK_BINARY,"housing":BANK_BINARY,"loan":BANK_BINARY,"y":BANK_BINARY},
   ranges={"age":(18,100),"duration":(0,5000),"balance":(-10000,200000),
           "day":(1,31),"campaign":(1,100)},
   not_null=[], sentinels=("unknown",)),
 "diabetes_130us": RuleSet("diabetes_130us",
   domains={"gender":DIAB_GENDER,"readmitted":DIAB_READMIT,"change":{"Ch","No"},
            "diabetesMed":{"Yes","No"}, **DIAB_DRUGS},
   ranges={"time_in_hospital":(1,14),"num_medications":(1,100),
           "num_lab_procedures":(0,200),"number_diagnoses":(1,20)},
   not_null=["race","weight","payer_code","medical_specialty"], sentinels=("?",)),
 "online_retail_ii": RuleSet("online_retail_ii",
   domains={}, ranges={"Quantity":(1,10000),"Price":(0.01,10000)},
   not_null=["Description","Customer ID"], sentinels=("?","","nan","None")),
}
for k,v in RULES.items(): print(f"{k:22s} {len(v.columns())} columns under rules")

In [ ]:
#@title SANITY CHECK -- run before trusting any table
# Define seeds locally so this cell works standalone, before the sweep cell.
SEEDS = seed_list(20260803, 5)

corrupted, gt = PLANS["bank_marketing_full"]().run(
    CLEAN["bank_marketing_full"], 0.10, SEEDS[0])
res = run_baselines(corrupted, RULES["bank_marketing_full"])

print("tools returned:", list(res))
for tool, r in res.items():
    n = "unavailable" if r["detections"] is None else len(r["detections"])
    print(f"  {tool:20s} {r['status'][:40]:40s} {n}")

det = res["Pandera"]["detections"]
overlap = len(set(det.row_id.astype(str)) & set(gt.row_id.astype(str)))
print("\ndetection ids :", det.row_id.head(2).tolist())
print("ground truth  :", gt.row_id.head(2).tolist())
print("row_id overlap:", overlap)

assert len(res) >= 4, "Fewer than 4 tools ran -- you are on a stub module."
assert overlap > 0, ("Zero row_id overlap: detections use positional indices "
                     "instead of _ztlf_row_id. Scores would be meaningless.")
assert len(det) > 20000, f"Only {len(det)} detections; rule set is incomplete."
print("\nSANITY CHECK PASSED")

In [ ]:
#@title Run every baseline -- RESUMABLE, memory-bounded
# Replaces the original sweep cell. Safe to re-run after a crash: completed
# (dataset, tool, rate, seed) combinations are skipped, not recomputed.
import gc, logging, os, time
import pandas as pd, numpy as np

# Great Expectations builds an ephemeral context and a temp docs directory on
# every validate() call. Across 75 sweep iterations these accumulate and
# exhaust Colab RAM. Quieten it and force collection between runs.
for noisy in ("great_expectations", "great_expectations.data_context"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

RATES   = [0.01, 0.05, 0.10, 0.20, 0.30]   #@param
N_SEEDS = 5                                 #@param {type:"integer"}
SEEDS   = seed_list(20260803, N_SEEDS)

CKPT = METRICS / "baseline_comparison_checkpoint.csv"
COLS = ["dataset","tool","rate","seed","status","seconds",
        "precision","recall","f1","n_detected"]

if CKPT.exists():
    done_df = pd.read_csv(CKPT)
    done = set(zip(done_df.dataset, done_df.tool,
                   done_df.rate.round(4), done_df.seed))
    print(f"resuming: {len(done_df)} rows already complete")
else:
    done_df = pd.DataFrame(columns=COLS)
    done = set()
    done_df.to_csv(CKPT, index=False)
    print("starting fresh")


def _append(rows):
    """Append immediately so a crash never costs more than one iteration."""
    pd.DataFrame(rows, columns=COLS).to_csv(CKPT, mode="a", header=False, index=False)


t0 = time.time()
for name, clean in CLEAN.items():
    rules = RULES[name]
    nat = natural_defect_keys_full(clean, SPECS[name], rules.columns())
    print(f"\n{name}: excluding {len(nat):,} naturally defective cells")

    for rate in RATES:
        for seed in SEEDS:
            pending = [t for t in BASELINES
                       if (name, t, round(rate, 4), seed) not in done]
            if not pending:
                continue

            corrupted, gt = PLANS[name]().run(clean, rate, seed)
            res = run_baselines(corrupted, rules, tools=pending)

            rows = []
            for tool, r in res.items():
                if r["detections"] is None:
                    rows.append(dict(dataset=name, tool=tool, rate=rate, seed=seed,
                                     status=r["status"], seconds=np.nan,
                                     precision=np.nan, recall=np.nan, f1=np.nan,
                                     n_detected=0))
                    continue
                o = score_restricted(gt, r["detections"], nat).attrs["overall"]
                rows.append(dict(dataset=name, tool=tool, rate=rate, seed=seed,
                                 status="ok", seconds=r["seconds"],
                                 precision=o["precision"], recall=o["recall"],
                                 f1=o["f1"], n_detected=len(r["detections"])))
            _append(rows)

            # release everything before the next iteration
            del corrupted, gt, res, rows
            gc.collect()

        print(f"  rate {rate}: done ({time.time()-t0:.0f}s)")

bench = pd.read_csv(CKPT)
bench.to_csv(METRICS / "baseline_comparison_raw.csv", index=False)
print(f"\n{len(bench)} rows total in {time.time()-t0:.0f}s")
print(bench.groupby(['dataset','tool']).size().to_string())

In [ ]:
#@title Table 6 — head-to-head tool comparison
# Ensure the 'bench' dataframe uses the new naming convention
if 'bench' in globals():
    bench['tool'] = bench['tool'].replace("ZTLF (ours)", "This work")

T6 = (bench[bench.status=="ok"]
      .groupby(["dataset","tool"])
      .agg(precision=("precision","mean"), recall=("recall","mean"),
           f1=("f1","mean"), f1_sd=("f1","std"),
           seconds=("seconds","mean"), detections=("n_detected","mean"))
      .round(4).reset_index())
T6.to_csv(TABLES/"T6_baseline_comparison.csv", index=False)
display(T6)

In [ ]:
#@title Table 7 — capability matrix (what each tool CAN express)
CAPABILITY = pd.DataFrame([
 # tool, domain, range, not-null, uniqueness, cross-field, cell-level output, in-memory
 ["This work",        "yes","yes","yes","yes","yes","yes","yes"],
 ["Pandera",          "yes","yes","yes","yes","yes","yes","yes"],
 ["GreatExpectations","yes","yes","yes","yes","partial","yes","yes"],
 ["SodaCore 4.x",     "yes","yes","yes","yes","yes","aggregate only","no (SQL source required)"],
 ["PyDeequ",          "yes","yes","yes","yes","partial","constraint level only","no (Spark required)"],
], columns=["tool","domain","range","not_null","uniqueness","cross_field",
            "per_cell_output","in_memory_dataframe"])
CAPABILITY.to_csv(TABLES/"T7_capability_matrix.csv", index=False)
print("Capability differences matter more than the accuracy ties.")
display(CAPABILITY)

In [ ]:
#@title Figure 4 — accuracy vs cost across tools
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

# Ensure the 'bench' dataframe uses the new naming convention globally
if 'bench' in globals():
    bench['tool'] = bench['tool'].replace("ZTLF (ours)", "This work")

ok = bench[bench.status=="ok"]
fig, axes = plt.subplots(1, 2, figsize=(13,4.6))

d = ok.groupby("tool").agg(f1=("f1","mean"), secs=("seconds","mean")).reset_index()
# Sorting to ensure consistent bar placement
d = d.sort_values("tool")

axes[0].bar(d.tool, d.f1); axes[0].set_ylabel("mean F1"); axes[0].set_ylim(0,1)
axes[0].set_title("Detection accuracy (identical rules)")
axes[0].tick_params(axis="x", rotation=20); axes[0].grid(axis="y", alpha=.3)
for i,v in enumerate(d.f1): axes[0].text(i, v+0.02, f"{v:.3f}", ha="center", fontsize=8)

axes[1].bar(d.tool, d.secs, color="tab:orange"); axes[1].set_ylabel("mean seconds")
axes[1].set_title("Runtime cost"); axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=.3)
for i,v in enumerate(d.secs):
    if pd.notna(v): axes[1].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)

fig.suptitle("Given the same rules, accuracy converges; cost and capability do not", y=1.03)
fig.tight_layout(); fig.savefig(FIGURES/"F4_baseline_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#@title Diagnostic — where do tools disagree? (this is the real finding)
import pandas as pd

# Ensure dependencies and definitions are available locally
DIAG_RATE = 0.10
N_SEEDS = 5
SEEDS = seed_list(20260803, N_SEEDS)

# Fallback: Re-import or ensure RULES exists
try:
    RULES
except NameError:
    # Minimal re-definition of rules if they are missing from scope
    DIAB_DRUGS = {c: DIAB_DRUG_LEVELS for c in ("metformin","insulin","glipizide","glyburide")}
    RULES = {
     "bank_marketing_full": RuleSet("bank_marketing_full",
       domains={"job":BANK_JOBS,"marital":BANK_MARITAL,"education":BANK_EDUCATION,
                "month":BANK_MONTH,"contact":BANK_CONTACT,"poutcome":BANK_POUTCOME,
                "default":BANK_BINARY,"housing":BANK_BINARY,"loan":BANK_BINARY,"y":BANK_BINARY},
       ranges={"age":(18,100),"duration":(0,5000),"balance":(-10000,200000),
               "day":(1,31),"campaign":(1,100)},
       not_null=[], sentinels=("unknown",)),
     "diabetes_130us": RuleSet("diabetes_130us",
       domains={"gender":DIAB_GENDER,"readmitted":DIAB_READMIT,"change":{"Ch","No"},
                "diabetesMed":{"Yes","No"}, **DIAB_DRUGS},
       ranges={"time_in_hospital":(1,14),"num_medications":(1,100),
               "num_lab_procedures":(0,200),"number_diagnoses":(1,20)},
       not_null=["race","weight","payer_code","medical_specialty"], sentinels=("?",)),
     "online_retail_ii": RuleSet("online_retail_ii",
       domains={}, ranges={"Quantity":(1,10000),"Price":(0.01,10000)},
       not_null=["Description","Customer ID"], sentinels=("?","","nan","None")),
    }

# The diagnostic loop
diag_rows = []
for name, clean in CLEAN.items():
    if name not in RULES:
        continue
    corrupted, gt = PLANS[name]().run(clean, DIAG_RATE, SEEDS[0])
    res = run_baselines(corrupted, RULES[name])
    per_col = {}
    for tool, r in res.items():
        if r["detections"] is None or len(r["detections"]) == 0:
            continue
        # Map tool name to 'This work' if necessary
        display_name = "This work" if tool == "ZTLF (ours)" else tool
        per_col[display_name] = r["detections"].groupby("column").size()

    if per_col:
        cmp = pd.DataFrame(per_col).fillna(0).astype(int)
        cmp["dataset"] = name
        diag_rows.append(cmp.reset_index().rename(columns={"index": "column"}))

disagree = pd.concat(diag_rows, ignore_index=True) if diag_rows else pd.DataFrame()
disagree.to_csv(TABLES / "T8_per_column_disagreement.csv", index=False)

print("Columns where tools disagree reveal capability gaps, e.g. Great "
      "Expectations' range expectations skip nulls, so type violations that "
      "coerce to NaN are never flagged.\n")
display(disagree)

---
### Interpreting this honestly

Accuracy parity is what I expected and what happened — my gate, Pandera, and Soda apply the same predicates to the same data and land on the same defects. That's the result, reported as such, no spin needed.

The real findings are in the differences:

- Great Expectations' range expectations don't fire on nulls. Type violations that coerce to NaN just pass silently, which costs recall. If you use GX, pair every range expectation with an explicit not-null check.
- Soda Core 4.x can't validate an in-memory DataFrame at all — it wants a SQL source and reports aggregate pass/fail rather than which rows failed, so per-cell detail means extra plumbing.
- PyDeequ only reports at constraint level, not per-cell, and needs a matching Spark/Java stack just to run. Real portability cost.
- Runtime differs by close to an order of magnitude for the exact same logical rules.

### Next: Phase 4
Does passing the quality gate actually help downstream model performance, and which defect classes matter most? Plus subgroup retention after quarantine — that's where the fairness question actually lives.
